In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from tqdm.auto import tqdm
import pyarrow.parquet as pq
import urllib.request

In [ ]:
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet'
local_file = 'yellow_tripdata_2021-01.parquet'

In [ ]:
urllib.request.urlretrieve(url, local_file)
print("Downloaded:", local_file)

Downloaded: yellow_tripdata_2021-01.parquet


In [ ]:
pf = pq.ParquetFile(local_file)

Rows: 1369769
Columns: 19
VendorID: int64
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: double
trip_distance: double
RatecodeID: double
store_and_fwd_flag: string
PULocationID: int64
DOLocationID: int64
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
airport_fee: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 2492


In [ ]:
pg_user = "root"
pg_pass = "root"
pg_host = "localhost"
pg_port = 5432
pg_db = "ny_taxi"

batchsize = 100000

target_table = 'yellow_taxi_data'

In [ ]:
engine = create_engine(f'postgresql+psycopg://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}')

In [ ]:
pf_iter = pf.iter_batches(batch_size=batchsize)

In [ ]:

first = True

for batch in tqdm(pf_iter, total=pf.num_row_groups):
    df_chunk = batch.to_pandas()

    if first:
        df_chunk.head(0).to_sql(name=target_table, con=engine, if_exists="replace")
        first = False
        print("Table created")

    df_chunk.to_sql(name=target_table, con=engine, if_exists="append")

    print("Inserted: ", len(df_chunk))

  0%|          | 0/1 [00:00<?, ?it/s]

Table created
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  100000
Inserted:  69769
